# Universally Slimmable ResNet-50 on CIFAR-10

Converts the fixed-width slimmable network into a **Universally Slimmable Network (US-Net)**:

- Widths are sampled **continuously** from `[min_width, max_width]`, not from a fixed list
- Trained with the **sandwich rule** (max width + min width + N random widths per step)
- Sub-networks are trained with **in-place distillation**: only the max-width network sees
  the true labels; every other sampled width is trained to match the max-width network's
  soft output in the same forward/backward pass
- Batch norm uses a single shared `USBatchNorm2d` per layer, statistics are **recalibrated**
  after training for each width you actually want to deploy

In [ ]:
# Clear any partial downloads and download the CIFAR-10 dataset tarball
!rm -rf ./data  # clear the partial download

!wget -q --show-progress https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz -O cifar10.tgz

In [ ]:
# 1. Extract the CIFAR-10 dataset tarball
!tar -xzf cifar10.tgz

# 2. Check the extracted folder structure
!ls cifar10
!ls cifar10/train

In [1]:
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim

## Config

`width_mult_list` is gone. In its place:

- `width_mult_range`: the continuous interval sub-networks are sampled from during training
- `num_sample_widths`: how many *extra* random widths to sample each step (sandwich rule
  always includes max + min on top of these)
- `bn_calibration_widths`: the specific widths you intend to deploy/evaluate at — BN stats
  get recalibrated for exactly these widths after training
- `width_divisor`: channel counts are rounded to multiples of this (standard practice, keeps
  channel counts hardware-friendly)

In [22]:
class DictToObj:
    def __init__(self, d):
        self.__dict__.update(d)

FLAGS = DictToObj({
    "num_gpus_per_job": 2,
    "num_cpus_per_job": 63,
    "memory_per_job": 380,
    "gpu_type": "nvidia-tesla-t4",
    "dataset": "cifar10",
    "num_classes": 10,
    "image_size": 32,
    "topk": [1, 5],
    "num_epochs": 100,
    "optimizer": "sgd",
    "momentum": 0.9,
    "weight_decay": 0.0001,
    "nesterov": True,
    "lr": 0.1,
    "lr_scheduler": "multistep",
    "multistep_lr_gamma": 0.1,
    "profiling": ["gpu"],
    "pretrained": "logs/us_resnet50_0.25_1.0.pt",
    "resume": "",
    "test_only": False,
    "random_seed": 1995,
    "batch_size": 256,
    "model": "models.us_resnet",
    "reset_parameters": True,
    "log_dir": "logs/",
    "slimmable_training": True,
    "depth": 50,

    # --- universally slimmable settings ---
    "width_mult_range": (0.25, 1.0),      # continuous training range [min, max]
    "num_sample_widths": 2,               # extra random (width, bit_width) pairs per sandwich step
    "width_divisor": 8,                   # round channel counts to a multiple of this

    # --- quantization-aware training settings ---
    "bit_width_list": [4, 8, 16, 32],     # 32 == full precision, always used for the teacher
    "deploy_configs": [
    (0.25, 4),
    (0.25, 8),
    (0.25, 16),
    (0.25, 32),

    (0.5, 4),
    (0.5, 8),
    (0.5, 16),
    (0.5, 32),

    (0.75, 4),
    (0.75, 8),
    (0.75, 16),
    (0.75, 32),

    (1.0, 4),
    (1.0, 8),
    (1.0, 16),
    (1.0, 32),
],
    "bn_calibration_widths":[0.25, 0.5, 0.75, 1.0]
})

# Multistep LR milestones scaled to num_epochs. The original hardcoded
# [30, 60, 90] silently never fired when num_epochs=5 -- the scheduler would
# step() every epoch but never actually hit a milestone.
FLAGS.multistep_lr_milestones = [int(FLAGS.num_epochs * 0.5), int(FLAGS.num_epochs * 0.75)]

In [3]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.transforms import v2

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((32, 32)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = ImageFolder("./cifar10/train", transform=train_transform)
test_dataset = ImageFolder("./cifar10/test", transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=FLAGS.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=FLAGS.batch_size,
    shuffle=False,       # no reason to shuffle eval data
    num_workers=2,
    pin_memory=True,
)

epochs = FLAGS.num_epochs

## Universally slimmable building blocks

`USConv2d` / `USLinear` / `USBatchNorm2d` compute their active channel count from
`self.width_mult * max_channels` at forward time (rounded via `make_divisible`), instead of
indexing into a fixed list. This is what makes *any* width in `width_mult_range` valid, not
just the four values you trained the old switchable-BN version on.

Two exceptions that are never slimmed:
- The stem conv's **input** is always 3 (RGB) channels — flagged with `is_stem=True`.
- The classifier's **output** is always `num_classes`.

`USBatchNorm2d` does *not* keep persistent running statistics during training (`training=True`
unconditionally, no momentum) — with a single shared BN, stats computed while width jumps
around every step aren't meaningful anyway. We recompute proper running stats per deployment
width after training with `recalibrate_bn` (further down).

In [ ]:
def make_divisible(v, divisor=8, min_value=None):
    # Round a channel count to the nearest multiple of `divisor`
    # (never rounding down by more than 10%).
    min_value = min_value or divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def fake_quantize_weight(w, bits, per_channel=True):
    """Symmetric fake quantization for weights, straight-through gradient estimator.
    bits >= 32 is treated as full precision (no-op)."""
    if bits >= 32:
        return w

    qmax = 2 ** (bits - 1) - 1  # symmetric range, e.g. bits=8 -> [-127, 127]

    if per_channel:
        # scale computed per output channel (dim 0) -- matches how we slice
        # channels per width_mult, so a sliced sub-network gets its own scale
        dims = tuple(range(1, w.dim()))
        max_val = w.detach().abs().amax(dim=dims, keepdim=True).clamp(min=1e-8)
    else:
        max_val = w.detach().abs().max().clamp(min=1e-8)

    scale = max_val / qmax
    w_q = torch.clamp(torch.round(w / scale), -qmax - 1, qmax) * scale

    # straight-through estimator: forward uses w_q, backward gradient
    # passes through as if this were the identity function
    return w + (w_q - w).detach()


def fake_quantize_act(x, bits):
    """Unsigned fake quantization for post-ReLU activations (always >= 0).
    Per-tensor, dynamic range (recomputed every forward from the current
    batch), straight-through gradient estimator. bits >= 32 is a no-op."""
    if bits >= 32:
        return x

    qmax = 2 ** bits - 1  # unsigned range, e.g. bits=8 -> [0, 255]
    max_val = x.detach().max().clamp(min=1e-8)
    scale = max_val / qmax
    x_q = torch.clamp(torch.round(x / scale), 0, qmax) * scale
    return x + (x_q - x).detach()


class ResnetConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True, is_stem=False):
        super(ResnetConv2d, self).__init__(
            in_channels, out_channels, kernel_size, stride=stride,
            padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.max_in_channels = in_channels
        self.max_out_channels = out_channels
        self.groups_ = groups
        self.is_stem = is_stem
        self.width_mult = max(FLAGS.width_mult_range)
        self.bit_width = max(FLAGS.bit_width_list)   # 32 = full precision by default

        # Only safe to enable once weights are frozen (see markdown above).
        self.quant_cache_enabled = False
        self._quant_cache = {}

    def clear_quant_cache(self):
        self._quant_cache = {}

    def _quantized_weight(self, weight):
        if self.bit_width >= 32:
            return weight
        if self.quant_cache_enabled:
            key = (self.width_mult, self.bit_width)
            if key in self._quant_cache:
                return self._quant_cache[key]
            wq = fake_quantize_weight(weight, self.bit_width)
            self._quant_cache[key] = wq
            return wq
        return fake_quantize_weight(weight, self.bit_width)

    def forward(self, input):
        in_ratio = 1.0 if self.is_stem else self.width_mult
        self.in_channels = make_divisible(self.max_in_channels * in_ratio, FLAGS.width_divisor)
        self.out_channels = make_divisible(self.max_out_channels * self.width_mult, FLAGS.width_divisor)
        self.groups = self.in_channels if self.groups_ != 1 else 1

        weight = self.weight[:self.out_channels, :self.in_channels, :, :]
        weight = self._quantized_weight(weight)  # quantization applied after slicing

        bias = self.bias[:self.out_channels] if self.bias is not None else None
        return nn.functional.conv2d(
            input, weight, bias, self.stride, self.padding,
            self.dilation, self.groups)


class ResnetConv(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super(ResnetConv, self).__init__(in_features, out_features, bias=bias)
        self.max_in_features = in_features
        self.max_out_features = out_features
        self.width_mult = max(FLAGS.width_mult_range)
        self.bit_width = max(FLAGS.bit_width_list)

        self.quant_cache_enabled = False
        self._quant_cache = {}

    def clear_quant_cache(self):
        self._quant_cache = {}

    def _quantized_weight(self, weight):
        if self.bit_width >= 32:
            return weight
        if self.quant_cache_enabled:
            key = (self.width_mult, self.bit_width)
            if key in self._quant_cache:
                return self._quant_cache[key]
            wq = fake_quantize_weight(weight, self.bit_width)
            self._quant_cache[key] = wq
            return wq
        return fake_quantize_weight(weight, self.bit_width)

    def forward(self, input):
        self.in_features = make_divisible(self.max_in_features * self.width_mult, FLAGS.width_divisor)
        self.out_features = self.max_out_features  # classifier output never slimmed

        weight = self.weight[:self.out_features, :self.in_features]
        weight = self._quantized_weight(weight)

        bias = self.bias if self.bias is not None else None
        return nn.functional.linear(input, weight, bias)


class ResnetBatchNorm2d(nn.BatchNorm2d):
    def __init__(self, num_features):
        super(ResnetBatchNorm2d, self).__init__(num_features, affine=True)
        self.max_features = num_features
        self.width_mult = max(FLAGS.width_mult_range)
        # BN itself is never quantized, but the *activation distribution*
        # feeding it shifts with the upstream conv's bit_width too, so we
        # track bit_width here purely to key the calibrated-stats dict --
        # kept in sync automatically by Model.set_bit_width().
        self.bit_width = max(FLAGS.bit_width_list)
        # only used post-training, by recalibrate_bn -- keyed by
        # (width_mult, bit_width) since both change the activation stats
        self.calibrated_running_mean = {}
        self.calibrated_running_var = {}

    def forward(self, input):
        c = make_divisible(self.max_features * self.width_mult, FLAGS.width_divisor)
        weight = self.weight[:c]
        bias = self.bias[:c]

        key = (self.width_mult, self.bit_width)
        if (not self.training) and key in self.calibrated_running_mean:
            # use recalibrated stats for this (width, bit_width) if we have them
            mean = self.calibrated_running_mean[key]
            var = self.calibrated_running_var[key]
            return nn.functional.batch_norm(
                input, mean, var, weight, bias, training=False, eps=self.eps)

        # training, or eval without calibration yet: batch statistics, no running average
        return nn.functional.batch_norm(
            input, None, None, weight, bias,
            training=True, momentum=0.0, eps=self.eps)


class ActQuant(nn.Module):
    """Fake-quantizes post-ReLU activations. Its bit_width is kept in sync
    with the conv/linear layers by Model.set_bit_width() (anything with a
    `bit_width` attribute gets updated together)."""
    def __init__(self):
        super().__init__()
        self.bit_width = max(FLAGS.bit_width_list)

    def forward(self, x):
        return fake_quantize_act(x, self.bit_width)


def clear_all_quant_caches(model):
    core = model.module if hasattr(model, 'module') else model
    for m in core.modules():
        if hasattr(m, 'clear_quant_cache'):
            m.clear_quant_cache()


def set_quant_cache_enabled(model, enabled):
    """Only call this with enabled=True for fixed-width/fixed-bitwidth
    inference after training -- never during training, since weights
    change every step and would leave the cache stale."""
    core = model.module if hasattr(model, 'module') else model
    for m in core.modules():
        if hasattr(m, 'quant_cache_enabled'):
            m.quant_cache_enabled = enabled
    if enabled:
        clear_all_quant_caches(model)

## Bottleneck block

Structurally identical to the original `Block`, just built once at max width — `USConv2d` /
`USBatchNorm2d` handle the slicing themselves at forward time, so there's no more need to pass
in a *list* of channel counts per block.

In [ ]:
class Block(nn.Module):
    def __init__(self, inp, outp, stride):
        super(Block, self).__init__()
        assert stride in [1, 2]

        midp = outp // 4
        self.conv1 = ResnetConv2d(inp, midp, 1, 1, 0, bias=False)
        self.bn1 = ResnetBatchNorm2d(midp)
        self.act1 = ActQuant()

        self.conv2 = ResnetConv2d(midp, midp, 3, stride, 1, bias=False)
        self.bn2 = ResnetBatchNorm2d(midp)
        self.act2 = ActQuant()

        self.conv3 = ResnetConv2d(midp, outp, 1, 1, 0, bias=False)
        self.bn3 = ResnetBatchNorm2d(outp)

        self.relu = nn.ReLU(inplace=True)

        self.residual_connection = stride == 1 and inp == outp
        if not self.residual_connection:
            self.shortcut_conv = ResnetConv2d(inp, outp, 1, stride=stride, bias=False)
            self.shortcut_bn = ResnetBatchNorm2d(outp)

        self.post_act = ActQuant()

    def forward(self, x):
        out = self.act1(self.relu(self.bn1(self.conv1(x))))
        out = self.act2(self.relu(self.bn2(self.conv2(out))))
        out = self.bn3(self.conv3(out))

        if self.residual_connection:
            res = x
        else:
            res = self.shortcut_bn(self.shortcut_conv(x))

        out = self.relu(out + res)
        out = self.post_act(out)
        return out

## Model

Every layer is now built once, at **max width** (`FLAGS.width_mult_range[1]`) — the US layers
scale themselves down at forward time based on whatever `width_mult` `set_width` last set.

In [ ]:
class Model(nn.Module):
    def __init__(self, num_classes=1000, input_size=224):
        super(Model, self).__init__()
        self.features = []

        self.block_setting_dict = {50: [3, 4, 6, 3], 101: [3, 4, 23, 3], 152: [3, 8, 36, 3]}
        self.block_setting = self.block_setting_dict[FLAGS.depth]
        feats = [64, 128, 256, 512]
        max_width = max(FLAGS.width_mult_range)
        channels = make_divisible(64 * max_width, FLAGS.width_divisor)

        if FLAGS.dataset == "cifar10":
            self.features.append(nn.Sequential(
                ResnetConv2d(3, channels, 3, 1, 1, bias=False, is_stem=True),
                ResnetBatchNorm2d(channels),
                nn.ReLU(inplace=True),
                ActQuant(),
            ))
        else:
            assert input_size % 32 == 0
            self.features.append(nn.Sequential(
                ResnetConv2d(3, channels, 7, 2, 3, bias=False, is_stem=True),
                ResnetBatchNorm2d(channels),
                nn.ReLU(inplace=True),
                ActQuant(),
                nn.MaxPool2d(3, 2, 1),
            ))

        # body -- unchanged structurally, blocks now carry activation quant internally
        for stage_id, n in enumerate(self.block_setting):
            outp = make_divisible(feats[stage_id] * max_width * 4, FLAGS.width_divisor)
            for i in range(n):
                stride = 2 if (i == 0 and stage_id != 0) else 1
                self.features.append(Block(channels, outp, stride))
                channels = outp

        self.features.append(nn.AdaptiveAvgPool2d(1))
        self.features = nn.Sequential(*self.features)

        self.outp = channels
        self.classifier = nn.Sequential(ResnetConv(self.outp, num_classes))
        if FLAGS.reset_parameters:
            self.reset_parameters()

    def forward(self, x):
        x = self.features(x)
        last_dim = x.size()[1]
        x = x.view(-1, last_dim)
        x = self.classifier(x)
        return x

    def set_width(self, width_mult):
        for m in self.modules():
            if hasattr(m, 'width_mult'):
                m.width_mult = width_mult

    def set_bit_width(self, bit_width):
        for m in self.modules():
            if hasattr(m, 'bit_width'):
                m.bit_width = bit_width

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                fan_out = m.weight.size(0) * m.kernel_size[0] * m.kernel_size[1]
                nn.init.normal_(m.weight, 0, math.sqrt(2. / fan_out))
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model(num_classes=FLAGS.num_classes, input_size=FLAGS.image_size)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

model = model.to(device)


def set_model_width(m, width_mult):
    # Works whether or not `m` is wrapped in DataParallel.
    (m.module if hasattr(m, 'module') else m).set_width(width_mult)


def set_model_bit_width(m, bit_width):
    (m.module if hasattr(m, 'module') else m).set_bit_width(bit_width)


set_model_width(model, max(FLAGS.width_mult_range))
set_model_bit_width(model, max(FLAGS.bit_width_list))

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(), lr=FLAGS.lr, momentum=FLAGS.momentum,
    weight_decay=FLAGS.weight_decay, nesterov=FLAGS.nesterov)
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=FLAGS.multistep_lr_milestones, gamma=FLAGS.multistep_lr_gamma)

Using 2 GPUs


## Training: sandwich rule + in-place distillation

Per batch:

1. Run the network at **max width** with the real labels (cross-entropy). This is the
   "teacher" for this step — its softmax output is saved (detached) as a soft target.
2. Run the network at **min width** and `num_sample_widths` random widths drawn uniformly
   from `width_mult_range`. These are trained only to match the teacher's soft output
   (KL divergence) — no hard labels for sub-networks, per the US-Net in-place distillation
   recipe.
3. All widths' gradients accumulate into the same `.grad` buffers (each `.backward()` adds to
   it) before a single `optimizer.step()`.

In [8]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    min_w, max_w = FLAGS.width_mult_range
    min_bw, max_bw = min(FLAGS.bit_width_list), max(FLAGS.bit_width_list)
    kd_loss_fn = nn.KLDivLoss(reduction='batchmean')
    total_loss, n_steps = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        # 1) max width, full precision -- real labels, this step's teacher
        set_model_width(model, max_w)
        set_model_bit_width(model, max_bw)
        out_max = model(images)
        loss_max = criterion(out_max, labels)
        loss_max.backward()
        soft_target = nn.functional.softmax(out_max.detach(), dim=1)

        # 2) sandwich rule: two guaranteed corners -- (min width, min bits)
        # and (max width, min bits) -- plus N random (width, bit_width)
        # pairs, all distilled from the teacher's soft output only.
        # The second corner exists because it's only reachable by chance
        # otherwise: width is drawn continuously and bit_width from a small
        # discrete set, so P(width landing near max_w AND bits landing on
        # min_bw in the same random draw) is low. Without a guaranteed
        # visit, (max_w, min_bw) gets far less training signal than
        # (min_w, min_bw) despite being an equally valid deployment point.
        sampled = [(min_w, min_bw), (max_w, min_bw)] + [
            (random.uniform(min_w, max_w), random.choice(FLAGS.bit_width_list))
            for _ in range(FLAGS.num_sample_widths)
        ]
        for wm, bw in sampled:
            set_model_width(model, wm)
            set_model_bit_width(model, bw)
            out = model(images)
            log_probs = nn.functional.log_softmax(out, dim=1)
            loss = kd_loss_fn(log_probs, soft_target)
            loss.backward()

        optimizer.step()
        total_loss += loss_max.item()
        n_steps += 1

    return total_loss / n_steps

## BN recalibration

Because `USBatchNorm2d` doesn't accumulate running statistics during training (batch stats
only), the model needs a short calibration pass **per width** before it can be evaluated in
`eval()` mode at that width. This runs a handful of forward-only batches through the network
at a fixed width, in `train()` mode (so `batch_norm` computes the batch statistics), and
stores a cumulative running mean/var into that width's slot on each `USBatchNorm2d`.

This only needs to be done once per width in `FLAGS.bn_calibration_widths`, e.g. right before
evaluation or deployment.

In [ ]:
@torch.no_grad()
def recalibrate_bn(model, loader, width_mult, bit_width, device, num_batches=100):
    core = model.module if hasattr(model, 'module') else model
    core.set_width(width_mult)
    core.set_bit_width(bit_width)

    key = (width_mult, bit_width)
    bn_layers = [m for m in core.modules() if isinstance(m, ResnetBatchNorm2d)]
    for m in bn_layers:
        m.calibrated_running_mean.pop(key, None)
        m.calibrated_running_var.pop(key, None)

    stats = {id(m): {'mean': None, 'var': None, 'n': 0} for m in bn_layers}

    def make_hook(m):
        def hook(module, inp, out):
            x = inp[0]
            dims = [0, 2, 3]
            batch_mean = x.mean(dim=dims)
            batch_var = x.var(dim=dims, unbiased=False)
            s = stats[id(m)]
            if s['mean'] is None:
                s['mean'], s['var'], s['n'] = batch_mean, batch_var, 1
            else:
                s['n'] += 1
                s['mean'] += (batch_mean - s['mean']) / s['n']
                s['var'] += (batch_var - s['var']) / s['n']
        return hook

    handles = [m.register_forward_hook(make_hook(m)) for m in bn_layers]

    try:  # guarantees cleanup on any error
        core.train()
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            core(images.to(device))
    finally:
        for h in handles:
            h.remove()

    for m in bn_layers:
        s = stats[id(m)]
        if s['mean'] is not None:
            m.calibrated_running_mean[key] = s['mean']
            m.calibrated_running_var[key] = s['var']

    core.eval()

In [10]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, configs=None):
    core = model.module if hasattr(model, 'module') else model  # unwrap here too
    configs = configs or FLAGS.deploy_configs
    core.eval()
    results = {}
    for wm, bw in configs:
        core.set_width(wm)
        core.set_bit_width(bw)
        correct, total, loss_sum = 0, 0, 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = core(images)       # single device
            loss_sum += criterion(outputs, labels).item()
            pred = outputs.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
        results[(wm, bw)] = {'loss': loss_sum / len(loader), 'acc': correct / total}
    return results

## Train

In [11]:
import os
import torch

start_epoch = 85
checkpoint_path = "us_resnet_epoch85_checkpoint.pt"  # Change to your checkpoint

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"]

    print(f"Resuming training from epoch {start_epoch}")
else:
    print("No checkpoint found. Starting from scratch.")

for epoch in range(start_epoch, FLAGS.num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    scheduler.step()

    print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }, f"us_resnet_epoch{epoch + 1}_checkpoint.pt")

Loading checkpoint: us_resnet_epoch85_checkpoint.pt
Resuming training from epoch 85
Epoch 86: train_loss=0.0175
Epoch 87: train_loss=0.0158
Epoch 88: train_loss=0.0150
Epoch 89: train_loss=0.0139
Epoch 90: train_loss=0.0148
Epoch 91: train_loss=0.0136
Epoch 92: train_loss=0.0139
Epoch 93: train_loss=0.0127
Epoch 94: train_loss=0.0130
Epoch 95: train_loss=0.0137
Epoch 96: train_loss=0.0120
Epoch 97: train_loss=0.0130
Epoch 98: train_loss=0.0128
Epoch 99: train_loss=0.0135
Epoch 100: train_loss=0.0128


## Recalibrate BN + evaluate at each deployment width

Run once after training finishes (or periodically, if you want to track sub-network accuracy
across epochs — recalibration is cheap, ~100 batches per width).

In [23]:
FLAGS.bn_calibration_widths = [0.25, 0.5, 0.75, 1.0]
FLAGS.bn_calibration_bits = [4, 8, 16, 32]

for wm in FLAGS.bn_calibration_widths:
    for bw in FLAGS.bn_calibration_bits:
        recalibrate_bn(model, train_loader, wm, bw, device, num_batches=100)

val_results = evaluate(model, test_loader, criterion, device)
for (wm, bw), r in val_results.items():
    print(f"width={wm} bits={bw}: loss={r['loss']:.4f} acc={r['acc']:.4f}")

width=0.25 bits=4: loss=0.3336 acc=0.8969
width=0.25 bits=8: loss=0.3085 acc=0.9046
width=0.25 bits=16: loss=0.3070 acc=0.9057
width=0.25 bits=32: loss=0.3073 acc=0.9059
width=0.5 bits=4: loss=0.3728 acc=0.9019
width=0.5 bits=8: loss=0.3137 acc=0.9183
width=0.5 bits=16: loss=0.3139 acc=0.9179
width=0.5 bits=32: loss=0.3137 acc=0.9183
width=0.75 bits=4: loss=0.3819 acc=0.9051
width=0.75 bits=8: loss=0.3437 acc=0.9212
width=0.75 bits=16: loss=0.3421 acc=0.9216
width=0.75 bits=32: loss=0.3426 acc=0.9218
width=1.0 bits=4: loss=0.3871 acc=0.9066
width=1.0 bits=8: loss=0.3611 acc=0.9232
width=1.0 bits=16: loss=0.3602 acc=0.9237
width=1.0 bits=32: loss=0.3607 acc=0.9237


In [20]:
core = model.module if hasattr(model, 'module') else model

core.set_bit_width(4)
w_full = core.features[0][0].weight.detach().clone()
w_quant = fake_quantize_weight(w_full, 4)
print((w_full - w_quant).abs().mean().item())
print(w_full.unique().numel(), w_quant.unique().numel())

0.02371923066675663
1728 562
